# **Study A** - **Human Gold-Ratings** and **LLM Evaluation** (3 Runs)

## Study A - Human Ratings

In [ ]:
!pip -q install pandas numpy

import os
import glob
import math
from dataclasses import dataclass
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

Mounted at /content/drive


### Config

In [ ]:
@dataclass
class CFG:
    # Drive paths
    stimuli_csv: str = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/studyA_dialogs_from_goodbad.csv"
    human_ratings_dir: str = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/StudyA_results"
    out_dir: str = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/studyA_exports"

    # Stimuli columns
    id_col: str = "dialog_id"
    text_col: str = "dialog_text"
    condition_col: str = "condition"     # good/bad
    is_gold_col: str = "is_gold"         # 0/1
    is_imc_col: str = "is_imc"           # 0/1

    # Human schema
    rater_col: str = "rater_id"

cfg = CFG()
os.makedirs(cfg.out_dir, exist_ok=True)

# Keep human column naming as in the CSVs
HUMAN_RATING_COLS = [
    "truthfulness", "relevance", "clarity", "logic_coherence",
    "feedback_depth", "relation_appropriateness", "respect_appreciation", "overall"
]

### I/O helpers

In [ ]:
def load_stimuli() -> pd.DataFrame:
    """Loads the final stimulus set (80 dialogs) and normalizes key columns."""
    df = pd.read_csv(cfg.stimuli_csv)
    required = [cfg.id_col, cfg.text_col, cfg.condition_col, cfg.is_gold_col, cfg.is_imc_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Stimuli missing required columns: {missing}")

    df[cfg.id_col] = df[cfg.id_col].astype(str).str.strip()
    df[cfg.condition_col] = df[cfg.condition_col].astype(str).str.strip().str.lower()
    df[cfg.is_gold_col] = pd.to_numeric(df[cfg.is_gold_col], errors="coerce").fillna(0).astype(int)
    df[cfg.is_imc_col] = pd.to_numeric(df[cfg.is_imc_col], errors="coerce").fillna(0).astype(int)
    return df


def load_all_human_ratings() -> pd.DataFrame:
    """Loads and concatenates all ratings_*.csv files into one raw table."""
    pattern = os.path.join(cfg.human_ratings_dir, "ratings_*.csv")
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No human rating files found with pattern: {pattern}")

    dfs = []
    for fp in files:
        df = pd.read_csv(fp)
        df["__source_file"] = os.path.basename(fp)
        dfs.append(df)

    out = pd.concat(dfs, ignore_index=True)
    out[cfg.id_col] = out[cfg.id_col].astype(str).str.strip()
    out[cfg.condition_col] = out[cfg.condition_col].astype(str).str.strip().str.lower()
    out[cfg.rater_col] = out[cfg.rater_col].astype(str).str.strip()
    return out


def _to_num(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    """Coerces selected columns to numeric (invalid values become NaN)."""
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

### Overview Human-Ratings

In [ ]:
def overview_human(df_h: pd.DataFrame, title: str) -> None:
    """Prints a compact overview of a human ratings table (raw or filtered)."""
    print(f"\n Human Overview: {title} ")
    print("Rows:", len(df_h))
    print("Raters:", df_h[cfg.rater_col].nunique() if cfg.rater_col in df_h.columns else "n/a")
    print("Dialogs:", df_h[cfg.id_col].nunique() if cfg.id_col in df_h.columns else "n/a")

    for c in ["is_gold", "is_imc", cfg.condition_col]:
        if c in df_h.columns:
            print(f"\n{c} counts:")
            print(df_h[c].value_counts(dropna=False))

    if "duration_sec" in df_h.columns:
        d = pd.to_numeric(df_h["duration_sec"], errors="coerce")
        print("\nDuration (sec) summary:")
        print(d.describe(percentiles=[.5, .9, .95, .99]))
        print("Rows > 300s:", int((d > 300).sum()))

    if all(c in df_h.columns for c in HUMAN_RATING_COLS):
        print("\nScale means (row-level):")
        print(df_h[HUMAN_RATING_COLS].apply(pd.to_numeric, errors="coerce").mean().round(3))

### Prereg QC - rater flags

In [ ]:
def add_gold_intervals_from_stimuli(df_h: pd.DataFrame, stimuli: pd.DataFrame) -> pd.DataFrame:
    """
    Merges preregistered gold-interval bounds (gold_expected_min/max) into human rows.
    This is used to flag gold expectation failures at rater level.
    """
    stim_cols = [cfg.id_col, "gold_expected_min", "gold_expected_max"]
    missing = [c for c in stim_cols if c not in stimuli.columns]
    if missing:
        raise ValueError(
            f"Stimuli missing prereg gold interval columns: {missing}. "
            "Expected: gold_expected_min, gold_expected_max."
        )

    stim = stimuli[stim_cols].copy()
    stim[cfg.id_col] = stim[cfg.id_col].astype(str).str.strip()
    stim["gold_expected_min"] = pd.to_numeric(stim["gold_expected_min"], errors="coerce")
    stim["gold_expected_max"] = pd.to_numeric(stim["gold_expected_max"], errors="coerce")

    out = df_h.copy()
    out[cfg.id_col] = out[cfg.id_col].astype(str).str.strip()

    for c in ["gold_expected_min", "gold_expected_max"]:
        if c in out.columns:
            out = out.drop(columns=[c])

    return out.merge(stim, on=cfg.id_col, how="left")


def compute_rater_flags_prereg(
    df_h_raw: pd.DataFrame,
    *,
    stimuli_for_gold_intervals: pd.DataFrame,
    expected_dialogs_per_rater: int = 10,
    expected_gold_per_rater: int = 1,
    expected_imc_per_rater: int = 1,
    min_median_sec: float = 20.0,
    lowvar_min_sd: float = 0.30,
    straightline_identical_share: float = 0.80,
) -> pd.DataFrame:
    """
    Computes preregistered rater-level QC flags.

    Flags (prereg):
    - flag_incomplete: wrong total/gold/imc counts vs planned assignment
    - flag_imc_fail: any IMC dialog has imc_dialog_pass == 0
    - flag_fast_median: median duration < 20s
    - flag_low_variance: SD across all scale responses < 0.30
    - flag_straightlining: max identical value share >= 0.80
    - flag_gold_fail: gold expectation outside prereg interval (overall outside bounds)
    """
    df = _to_num(
        df_h_raw,
        ["is_gold", "is_imc", "imc_dialog_pass", "duration_sec", *HUMAN_RATING_COLS],
    ).copy()

    # gold bounds are stored in stimuli
    df = add_gold_intervals_from_stimuli(df, stimuli_for_gold_intervals)
    df["overall_num"] = pd.to_numeric(df["overall"], errors="coerce")
    df["gold_expected_min"] = pd.to_numeric(df["gold_expected_min"], errors="coerce")
    df["gold_expected_max"] = pd.to_numeric(df["gold_expected_max"], errors="coerce")

    # counts per rater
    n_total = df.groupby(cfg.rater_col)[cfg.id_col].count()
    n_gold  = df.loc[df["is_gold"] == 1].groupby(cfg.rater_col)[cfg.id_col].count()
    n_imc   = df.loc[df["is_imc"] == 1].groupby(cfg.rater_col)[cfg.id_col].count()

    # IMC fail
    imc_fail = (
        df.loc[df["is_imc"] == 1]
        .groupby(cfg.rater_col)["imc_dialog_pass"]
        .apply(lambda s: (pd.to_numeric(s, errors="coerce") == 0).any())
    )

    # median time
    med_time = df.groupby(cfg.rater_col)["duration_sec"].median()

    # variance / straightlining (flatten all rating cells)
    stats_rows = []
    for rid, g in df.groupby(cfg.rater_col):
        vals = g[HUMAN_RATING_COLS].to_numpy(dtype=float).ravel()
        vals = vals[~np.isnan(vals)]
        if len(vals) == 0:
            sd_all = np.nan
            max_share = np.nan
        else:
            sd_all = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
            counts = pd.Series(vals).value_counts(normalize=True)
            max_share = float(counts.max()) if not counts.empty else np.nan
        stats_rows.append({cfg.rater_col: rid, "sd_all": sd_all, "max_share": max_share})
    patt = pd.DataFrame(stats_rows).set_index(cfg.rater_col)

    # gold fail (outside interval)
    def _gold_outside_interval(g: pd.DataFrame) -> bool:
        gg = g[g["is_gold"] == 1].copy()
        if gg.empty:
            return False
        m = gg["overall_num"].notna() & gg["gold_expected_min"].notna() & gg["gold_expected_max"].notna()
        gg = gg[m]
        if gg.empty:
            return False
        outside = (gg["overall_num"] < gg["gold_expected_min"]) | (gg["overall_num"] > gg["gold_expected_max"])
        return bool(outside.any())

    gold_fail = df.groupby(cfg.rater_col).apply(_gold_outside_interval)

    # assemble flag table
    all_raters = sorted(df[cfg.rater_col].astype(str).unique())
    flags = pd.DataFrame({cfg.rater_col: all_raters}).set_index(cfg.rater_col)

    def _get(series, rid, default):
        try:
            return series.get(rid, default)
        except Exception:
            return default

    flags["flag_incomplete"] = flags.index.map(
        lambda r: int(
            (_get(n_total, r, 0) != expected_dialogs_per_rater)
            or (_get(n_gold,  r, 0) != expected_gold_per_rater)
            or (_get(n_imc,   r, 0) != expected_imc_per_rater)
        )
    )
    flags["flag_imc_fail"] = flags.index.map(lambda r: bool(_get(imc_fail, r, False)))
    flags["flag_fast_median"] = flags.index.map(lambda r: float(_get(med_time, r, np.inf)) < float(min_median_sec))
    flags["flag_low_variance"] = flags.index.map(lambda r: float(_get(patt["sd_all"], r, np.inf)) < float(lowvar_min_sd))
    flags["flag_straightlining"] = flags.index.map(
        lambda r: float(_get(patt["max_share"], r, 0.0)) >= float(straightline_identical_share)
    )
    flags["flag_gold_fail"] = flags.index.map(lambda r: bool(_get(gold_fail, r, False)))

    return flags.reset_index()

### Row-level audit (include/exclude + reasons)

In [ ]:
def build_row_level_audit(
    df_h_raw: pd.DataFrame,
    *,
    include_flagged_participants: bool,
    include_gold_fail: bool,
    include_long_duration_rows: bool,
    long_duration_sec: float = 300.0,
) -> pd.DataFrame:
    """
    Creates an audit row for every rating row, including:
    - keep_row (given the PRIMARY/SENS toggles)
    - exclusion_reasons (comma-separated)
    """
    stimuli = load_stimuli()
    df = _to_num(df_h_raw, ["is_gold", "is_imc", "imc_dialog_pass", "duration_sec", *HUMAN_RATING_COLS]).copy()

    # add prereg flags
    flags = compute_rater_flags_prereg(df, stimuli_for_gold_intervals=stimuli)
    df = df.merge(flags, on=cfg.rater_col, how="left")

    # regular rows only drive confirmatory datasets
    df["is_regular"] = (df["is_gold"] == 0) & (df["is_imc"] == 0)

    # row flags
    df["flag_long_duration"] = pd.to_numeric(df["duration_sec"], errors="coerce") > float(long_duration_sec)
    df["flag_missing_scales"] = df[HUMAN_RATING_COLS].apply(pd.to_numeric, errors="coerce").isna().any(axis=1)

    # participant-level "core" exclusion flags (gold_fail is handled separately as prereg sensitivity toggle)
    df["flag_rater_excluded_core"] = (
        (df["flag_incomplete"] == 1)
        | (df["flag_imc_fail"] == True)
        | (df["flag_fast_median"] == True)
        | (df["flag_low_variance"] == True)
        | (df["flag_straightlining"] == True)
    )
    df["flag_rater_gold_fail"] = (df["flag_gold_fail"] == True)

    # keep logic (prereg)
    keep = df["is_regular"].copy()
    keep &= ~df["flag_missing_scales"]  # listwise deletion on required scales

    if not include_long_duration_rows:
        keep &= ~df["flag_long_duration"]

    if not include_flagged_participants:
        keep &= ~df["flag_rater_excluded_core"]

    if not include_gold_fail:
        keep &= ~df["flag_rater_gold_fail"]

    df["keep_row"] = keep.astype(bool)

    def _reasons(row) -> str:
        reasons = []
        if not bool(row["is_regular"]):
            if int(row.get("is_gold", 0)) == 1: reasons.append("non_regular_gold")
            if int(row.get("is_imc", 0)) == 1: reasons.append("non_regular_imc")
        if bool(row.get("flag_missing_scales", False)):
            reasons.append("missing_scales_listwise")
        if bool(row.get("flag_long_duration", False)) and not include_long_duration_rows:
            reasons.append("duration_gt_300s_primary_excluded")
        if bool(row.get("flag_rater_excluded_core", False)) and not include_flagged_participants:
            reasons.append("rater_excluded_core_flags")
        if bool(row.get("flag_rater_gold_fail", False)) and not include_gold_fail:
            reasons.append("rater_gold_fail_primary_excluded")
        return ", ".join(reasons)

    df["exclusion_reasons"] = df.apply(_reasons, axis=1)

    cols = [
        cfg.rater_col, cfg.id_col, cfg.condition_col,
        "is_gold", "is_imc", "is_regular",
        "duration_sec", "flag_long_duration", "flag_missing_scales",
        "flag_incomplete", "flag_imc_fail", "flag_fast_median", "flag_low_variance", "flag_straightlining", "flag_gold_fail",
        "keep_row", "exclusion_reasons",
        "__source_file",
    ]
    cols = [c for c in cols if c in df.columns]
    return df[cols].copy()

### Dialog-level human gold (mean of 2 raters)

In [ ]:
def enforce_exactly_two_raters_per_dialog(df_rows_kept: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Keeps only dialogs with exactly two unique raters (prereg human gold standard requirement)."""
    cov = (
        df_rows_kept.groupby(cfg.id_col)[cfg.rater_col]
        .nunique()
        .reset_index()
        .rename(columns={cfg.rater_col: "n_raters"})
    )

    if (cov["n_raters"] > 2).any():
        bad = cov.loc[cov["n_raters"] > 2, cfg.id_col].astype(str).tolist()
        raise ValueError(f"Found dialogs with >2 raters after filtering. Examples: {bad[:10]}")

    keep = set(cov.loc[cov["n_raters"] == 2, cfg.id_col])
    return df_rows_kept[df_rows_kept[cfg.id_col].isin(keep)].copy(), cov


def build_human_gold_dialog_level(df_rows_kept_2raters: pd.DataFrame) -> pd.DataFrame:
    """Computes the dialog-level human gold score as the mean of the two raters (per scale)."""
    mapping = {
        "truthfulness": "truthfulness",
        "relevance": "relevance",
        "clarity": "clarity",
        "logic_coherence": "logic_coherence",
        "feedback_depth": "feedback_depth",
        "relation_appropriateness": "relational_appropriateness",
        "respect_appreciation": "respect_appreciation",
        "overall": "overall_quality",
    }

    df = df_rows_kept_2raters.copy()
    for src in mapping.keys():
        df[src] = pd.to_numeric(df[src], errors="coerce")

    tmp = df[[cfg.id_col] + list(mapping.keys())].copy()
    agg = tmp.groupby(cfg.id_col).mean(numeric_only=True).reset_index()

    # rename to human_gold_*
    agg = agg.rename(columns={mapping[src]: f"human_gold_{mapping[src]}" for src in mapping.keys()})

    n_raters = (
        df.groupby(cfg.id_col)[cfg.rater_col]
        .nunique()
        .reset_index()
        .rename(columns={cfg.rater_col: "n_human_raters"})
    )

    return agg.merge(n_raters, on=cfg.id_col, how="left")

### Rater and dialog audit summary

In [ ]:
FLAG_COLS = [
    "flag_incomplete",
    "flag_imc_fail",
    "flag_fast_median",
    "flag_low_variance",
    "flag_straightlining",
    "flag_gold_fail",
]

def build_rater_audit_from_rows_audit(df_rows_audit: pd.DataFrame) -> pd.DataFrame:
    """Summarizes QC status and reasons per rater."""
    kept_raters = set(df_rows_audit[df_rows_audit["keep_row"]][cfg.rater_col].unique())

    rater_flags = (
        df_rows_audit.groupby(cfg.rater_col)[FLAG_COLS]
        .max()
        .reset_index()
    )

    def compact_reason(row):
        reasons = []
        if row["flag_incomplete"] == 1: reasons.append("incomplete_or_wrong_gold/imc_count")
        if row["flag_imc_fail"] == True: reasons.append("imc_fail")
        if row["flag_fast_median"] == True: reasons.append("median_time_lt_20s")
        if row["flag_low_variance"] == True: reasons.append("low_variance_sd_lt_0.30")
        if row["flag_straightlining"] == True: reasons.append("straightlining_share_ge_0.80")
        if row["flag_gold_fail"] == True: reasons.append("gold_expectation_fail")
        return ", ".join(reasons)

    rater_flags["reason_compact"] = rater_flags.apply(compact_reason, axis=1)
    rater_flags["status"] = rater_flags[cfg.rater_col].apply(lambda r: "KEPT" if r in kept_raters else "EXCLUDED")
    return rater_flags.sort_values(["status", cfg.rater_col])


def build_dialog_audit_from_rows_audit(df_rows_audit: pd.DataFrame) -> pd.DataFrame:
    """Summarizes inclusion/exclusion reasons at dialog level for the prereg confirmatory set."""
    stimuli = load_stimuli()
    stim_meta = stimuli[[cfg.id_col, cfg.condition_col, cfg.is_gold_col, cfg.is_imc_col]].copy()
    stim_meta = stim_meta.rename(columns={cfg.is_gold_col: "stim_is_gold", cfg.is_imc_col: "stim_is_imc"})

    kept_rows = df_rows_audit[df_rows_audit["keep_row"]].copy()

    cov = (
        kept_rows.groupby(cfg.id_col)[cfg.rater_col]
        .nunique()
        .reset_index()
        .rename(columns={cfg.rater_col: "n_valid_raters"})
    )

    excluded_rows = df_rows_audit[~df_rows_audit["keep_row"]].copy()
    per_dialog_reasons = (
        excluded_rows.groupby(cfg.id_col)["exclusion_reasons"]
        .apply(lambda s: sorted(set([x for x in s.tolist() if isinstance(x, str) and x.strip()])))
        .reset_index()
        .rename(columns={"exclusion_reasons": "row_exclusion_reasons"})
    )

    all_dialogs = pd.DataFrame({cfg.id_col: sorted(stim_meta[cfg.id_col].astype(str).unique())})
    dlg = all_dialogs.merge(stim_meta, on=cfg.id_col, how="left")
    dlg = dlg.merge(cov, on=cfg.id_col, how="left")
    dlg = dlg.merge(per_dialog_reasons, on=cfg.id_col, how="left")

    dlg["n_valid_raters"] = dlg["n_valid_raters"].fillna(0).astype(int)
    dlg["row_exclusion_reasons"] = dlg["row_exclusion_reasons"].apply(lambda x: x if isinstance(x, list) else [])

    dlg["is_regular_stimulus"] = (dlg["stim_is_gold"].fillna(0).astype(int) == 0) & (dlg["stim_is_imc"].fillna(0).astype(int) == 0)
    dlg["kept_for_human_gold"] = dlg["is_regular_stimulus"] & (dlg["n_valid_raters"] == 2)

    def compact_dialog_reason(row):
        reasons = []
        if not bool(row["is_regular_stimulus"]):
            if int(row.get("stim_is_gold", 0)) == 1: reasons.append("stimulus_is_gold")
            if int(row.get("stim_is_imc", 0)) == 1: reasons.append("stimulus_is_imc")
        if int(row.get("n_valid_raters", 0)) < 2 and bool(row["is_regular_stimulus"]):
            reasons.append("missing_second_valid_rater_after_qc")
        for r in row.get("row_exclusion_reasons", []):
            if r and r not in reasons:
                reasons.append(r)
        return ", ".join(reasons)

    dlg["reason_compact"] = dlg.apply(compact_dialog_reason, axis=1)
    return dlg

### Build and export datasets (PRIMARY vs. SENS)

In [ ]:
def build_and_export_human_datasets() -> Dict[str, str]:
    """
    Runs the prereg inclusion/exclusion pipeline and export:
    - PRIMARY rows kept (regular only) + human_gold dialog-level (2 raters only)
    - SENS rows kept (regular only) + human_gold dialog-level
      (gold_fail + flagged + >300s included as prereg sensitivity toggles)
    Plus audit tables (row/rater/dialog) for transparency.
    """
    df_h_raw = load_all_human_ratings()
    overview_human(df_h_raw, "RAW (all rows: regular + gold + IMC)")

    exports = {}

    # Primary (confirmatory): exclude flagged participants, exclude gold-fail, exclude >300s
    rows_audit_primary = build_row_level_audit(
        df_h_raw,
        include_flagged_participants=False,
        include_gold_fail=False,
        include_long_duration_rows=False,
        long_duration_sec=300.0,
    )

    # Keep only the rows that are in the confirmatory dataset
    df_primary_rows = df_h_raw.merge(
        rows_audit_primary[[cfg.rater_col, cfg.id_col, "keep_row", "exclusion_reasons"]],
        on=[cfg.rater_col, cfg.id_col],
        how="left"
    )
    df_primary_rows = df_primary_rows[df_primary_rows["keep_row"] == True].copy()

    # Enforce exactly 2 raters per dialog for human gold
    df_primary_rows_2, cov_primary = enforce_exactly_two_raters_per_dialog(df_primary_rows)
    human_gold_primary = build_human_gold_dialog_level(df_primary_rows_2)

    # Audits
    rater_audit_primary = build_rater_audit_from_rows_audit(rows_audit_primary)
    dialog_audit_primary = build_dialog_audit_from_rows_audit(rows_audit_primary)

    # Save PRIMARY
    p1 = os.path.join(cfg.out_dir, "human_PRIMARY_rows_regular_only_kept.csv")
    p2 = os.path.join(cfg.out_dir, "human_PRIMARY_gold_dialog_level.csv")
    p3 = os.path.join(cfg.out_dir, "audit_PRIMARY_row_level.csv")
    p4 = os.path.join(cfg.out_dir, "audit_PRIMARY_rater_level.csv")
    p5 = os.path.join(cfg.out_dir, "audit_PRIMARY_dialog_level.csv")

    df_primary_rows.to_csv(p1, index=False, encoding="utf-8")
    human_gold_primary.to_csv(p2, index=False, encoding="utf-8")
    rows_audit_primary.to_csv(p3, index=False, encoding="utf-8")
    rater_audit_primary.to_csv(p4, index=False, encoding="utf-8")
    dialog_audit_primary.to_csv(p5, index=False, encoding="utf-8")

    exports.update({
        "PRIMARY_rows": p1,
        "PRIMARY_human_gold": p2,
        "PRIMARY_audit_rows": p3,
        "PRIMARY_audit_raters": p4,
        "PRIMARY_audit_dialogs": p5,
    })

    # SENS: include flagged + include gold-fail + include >300s
    rows_audit_sens = build_row_level_audit(
        df_h_raw,
        include_flagged_participants=True,
        include_gold_fail=True,
        include_long_duration_rows=True,
        long_duration_sec=300.0,
    )

    df_sens_rows = df_h_raw.merge(
        rows_audit_sens[[cfg.rater_col, cfg.id_col, "keep_row", "exclusion_reasons"]],
        on=[cfg.rater_col, cfg.id_col],
        how="left"
    )
    df_sens_rows = df_sens_rows[df_sens_rows["keep_row"] == True].copy()

    df_sens_rows_2, cov_sens = enforce_exactly_two_raters_per_dialog(df_sens_rows)
    human_gold_sens = build_human_gold_dialog_level(df_sens_rows_2)

    rater_audit_sens = build_rater_audit_from_rows_audit(rows_audit_sens)
    dialog_audit_sens = build_dialog_audit_from_rows_audit(rows_audit_sens)

    # Save sensitivity (SENS)
    s1 = os.path.join(cfg.out_dir, "human_SENS_rows_regular_only_kept.csv")
    s2 = os.path.join(cfg.out_dir, "human_SENS_gold_dialog_level.csv")
    s3 = os.path.join(cfg.out_dir, "audit_SENS_row_level.csv")
    s4 = os.path.join(cfg.out_dir, "audit_SENS_rater_level.csv")
    s5 = os.path.join(cfg.out_dir, "audit_SENS_dialog_level.csv")

    df_sens_rows.to_csv(s1, index=False, encoding="utf-8")
    human_gold_sens.to_csv(s2, index=False, encoding="utf-8")
    rows_audit_sens.to_csv(s3, index=False, encoding="utf-8")
    rater_audit_sens.to_csv(s4, index=False, encoding="utf-8")
    dialog_audit_sens.to_csv(s5, index=False, encoding="utf-8")

    exports.update({
        "SENS_rows": s1,
        "SENS_human_gold": s2,
        "SENS_audit_rows": s3,
        "SENS_audit_raters": s4,
        "SENS_audit_dialogs": s5,
    })

    # Quick console summary (no hypothesis tests)
    print("\n Export Summary (Human) ")
    print("Primary dialogs (human gold):", human_gold_primary[cfg.id_col].nunique())
    print("Sensitivity dialogs (human gold):   ", human_gold_sens[cfg.id_col].nunique())
    print("Saved files to:", cfg.out_dir)

    return exports

### RUN (Human)

In [ ]:
# generate all human exports + audits
human_exports = build_and_export_human_datasets()
human_exports


 Human Overview: RAW (all rows: regular + gold + IMC) 
Rows: 156
Raters: 16
Dialogs: 80

is_gold counts:
is_gold
0    127
1     29
Name: count, dtype: int64

is_imc counts:
is_imc
0    137
1     19
Name: count, dtype: int64

condition counts:
condition
bad     78
good    78
Name: count, dtype: int64

Duration (sec) summary:
count    156.000000
mean     119.459995
std       58.033214
min       38.389532
50%      101.429457
90%      211.353610
95%      243.034365
99%      287.836171
max      298.174970
Name: duration_sec, dtype: float64
Rows > 300s: 0

Scale means (row-level):
truthfulness                4.250
relevance                   4.205
clarity                     4.981
logic_coherence             3.929
feedback_depth              3.962
relation_appropriateness    4.224
respect_appreciation        4.122
overall                     3.962
dtype: float64


/tmp/ipython-input-342109776.py:104: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  gold_fail = df.groupby(cfg.rater_col).apply(_gold_outside_interval)
/tmp/ipython-input-342109776.py:104: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  gold_fail = df.groupby(cfg.rater_col).apply(_gold_outside_interval)



 Export Summary (Human) 
Primary dialogs (human gold): 50
Sensitivity dialogs (human gold):    54
Saved files to: /content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/studyA_exports


{'PRIMARY_rows': '/content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/studyA_exports/human_PRIMARY_rows_regular_only_kept.csv',
 'PRIMARY_human_gold': '/content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/studyA_exports/human_PRIMARY_gold_dialog_level.csv',
 'PRIMARY_audit_rows': '/content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/studyA_exports/audit_PRIMARY_row_level.csv',
 'PRIMARY_audit_raters': '/content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/studyA_exports/audit_PRIMARY_rater_level.csv',
 'PRIMARY_audit_dialogs': '/content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/studyA_exports/audit_PRIMARY_dialog_level.csv',
 'SENS_rows': '/content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/studyA_exports/human_SENS_rows_regular_only_kept.csv',
 'SENS_human_gold': '/content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/studyA_exports/human_SENS_gold_dialog_level.csv',
 'SENS_audit_rows': '/content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/studyA_exports/audit_SENS_row_level.csv',
 'SENS_audit_rater

## Study A - LLM Evaluation (Post-hoc, 3 runs, median)

In [ ]:
import os
import json
import time
import re
import math
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Tuple, Dict
from openai import OpenAI
from openai import RateLimitError, APIError, APITimeoutError, APIConnectionError

### LLM config

In [ ]:
@dataclass
class LLM_CFG:
    base_url: str = "https://chat-ai.academiccloud.de/v1"
    api_key: str = "" # SAIA_API KEY
    model_name: str = "llama-3.3-70b-instruct"

    n_runs: int = 3
    temperatures: Tuple[float, float, float] = (0.2, 0.2, 0.2)

    max_tokens: int = 400
    max_retries: int = 4
    retry_sleep_sec: float = 2.0

llm_cfg = LLM_CFG()

# make retries more robust against API rate limits
llm_cfg.max_retries = 10
llm_cfg.retry_sleep_sec = 3.0

# small global throttle between successful requests
GLOBAL_THROTTLE_SEC = 0.4

# scale keys (LLM output schema)
SCALE_KEYS = [
    "clarity",
    "relevance",
    "truthfulness",
    "logic_coherence",
    "respect_appreciation",
    "relational_appropriateness",
    "feedback_depth",
    "overall_quality",
]

### LLM Prompt and parsing

In [ ]:
# communication-expert prompt (same as in Generation-Notebooks)
# Statements identical to statements in Human Study A
def create_communication_expert_prompt(discussion: str) -> str:
    """
    Communication expert prompt using the same Likert-style statements
    (1 = strongly disagree ... 7 = strongly agree)
    as human raters in Study A.
    """

    return f"""
You are a communication expert. Your task is to evaluate the COMMUNICATION QUALITY
of a dialogue between two nutrition experts. Focus ONLY on the communication,
not on nutritional correctness.

You will rate the dialogue on seven dimensions and one overall score,
using the SAME type of Likert-style statements that human raters in the study use.

GENERAL SCALE (applies to all items):
1 = strongly disagree
2 = disagree
3 = somewhat disagree
4 = neither agree nor disagree
5 = somewhat agree
6 = agree
7 = strongly agree

Rate the following eight statements:

1) CLARITY
"The dialogue is clear, easy to understand, and expressed in unambiguous language."

2) RELEVANCE
"Each turn in the dialogue responds meaningfully to the previous one and stays on topic."

3) TRUTHFULNESS
"The dialogue is internally consistent, without contradictions or misleading statements."

4) LOGIC / COHERENCE
"The dialogue presents coherent reasoning, with arguments that follow a logical structure."

5) RESPECT / APPRECIATION
"The dialogue maintains a respectful tone, acknowledging the partner’s contributions appropriately."

6) RELATIONAL APPROPRIATENESS
"The dialogue handles the interpersonal relationship appropriately, without tension, dominance, or relational violations."

7) FEEDBACK / DEPTH
"The dialogue builds meaningfully on the partner’s statements and contains sufficient conversational depth."

8) OVERALL QUALITY
"Overall, the dialogue shows high communication quality."

Here is the dialogue:
{discussion}

Important rating instructions:
- Assign an integer from 1 to 7 for each of the eight statements.
- Use the FULL RANGE of the 1–7 scale whenever appropriate.
- Do NOT collapse ratings toward the middle or top of the scale.
- Do not cluster ratings in the mid-range. Use extreme values (1 or 7) whenever clearly justified.
- The score “7” must be reserved for exceptional, outstanding communication quality that clearly stands out.
- The score “1” must be used for extremely poor communication with severe flaws.
- Most dialogues should fall BETWEEN these extremes (2–6), depending on their strengths and weaknesses.
- Evaluate each dimension independently based on the actual content of the dialogue.

Return ONLY a strict JSON object (no commentary, no code blocks), in this format:

{{
  "clarity": 1-7,
  "relevance": 1-7,
  "truthfulness": 1-7,
  "logic_coherence": 1-7,
  "respect_appreciation": 1-7,
  "relational_appropriateness": 1-7,
  "feedback_depth": 1-7,
  "overall_quality": 1-7
}}

Return only this JSON object with plain integers.
""".strip()

_JSON_BLOCK_RE = re.compile(r"\{.*\}", re.DOTALL)

def parse_llm_json(text: str) -> Dict[str, int]:
    """Extracts and validates the JSON object (exact keys, integer 1–7)."""
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Empty LLM response")

    m = _JSON_BLOCK_RE.search(text.strip())
    if not m:
        raise ValueError(f"No JSON object found. Head: {text[:200]}")

    obj = json.loads(m.group(0))
    if set(obj.keys()) != set(SCALE_KEYS):
        missing = sorted(set(SCALE_KEYS) - set(obj.keys()))
        extra = sorted(set(obj.keys()) - set(SCALE_KEYS))
        raise ValueError(f"Key mismatch. Missing={missing}, Extra={extra}")

    out = {}
    for k in SCALE_KEYS:
        v = obj[k]
        if v is None or isinstance(v, bool):
            raise ValueError(f"Invalid value for {k}: {v}")
        vi = int(v)
        if vi < 1 or vi > 7:
            raise ValueError(f"Out of range for {k}: {vi}")
        out[k] = vi
    return out

### Client and single run

In [ ]:
if not llm_cfg.api_key:
    llm_cfg.api_key = os.environ.get("OPENAI_API_KEY", "").strip()

if not llm_cfg.api_key:
    raise ValueError("Missing API key. Set OPENAI_API_KEY or llm_cfg.api_key.")

client = OpenAI(api_key=llm_cfg.api_key, base_url=llm_cfg.base_url)

def call_llm(prompt: str, temperature: float) -> str:
    """Call thes OpenAI-compatible endpoint and returns raw text."""
    resp = client.chat.completions.create(
        model=llm_cfg.model_name,
        temperature=float(temperature),
        max_tokens=int(llm_cfg.max_tokens),
        messages=[
            {"role": "system", "content": "You are a communication expert."},
            {"role": "user", "content": prompt},
        ],
    )
    return resp.choices[0].message.content or ""


def llm_rate_one(dialog_text: str, dialog_id: str, run_id: int) -> Tuple[Dict[str, int], str]:
    """Runs one inference with retries and stricts JSON parsing (robust to 429 rate limits)."""
    prompt = create_communication_expert_prompt(dialog_text)
    last_err = None

    base = float(llm_cfg.retry_sleep_sec)

    for attempt in range(1, llm_cfg.max_retries + 1):
        try:
            raw = call_llm(prompt, temperature=llm_cfg.temperatures[run_id - 1])
            scores = parse_llm_json(raw)

            # gentle throttle to reduce future 429s
            try:
                time.sleep(float(GLOBAL_THROTTLE_SEC))
            except NameError:
                pass

            return scores, raw

        except RateLimitError as e:
            last_err = e
            # exponential backoff + jitter, cap at 120s
            sleep_s = min(120.0, base * (2 ** (attempt - 1)) + float(np.random.uniform(0, 1.5)))
            print(f"RATE LIMIT 429 dialog={dialog_id} run={run_id} attempt={attempt}/{llm_cfg.max_retries} -> sleep {sleep_s:.1f}s")
            time.sleep(sleep_s)

        except (APITimeoutError, APIConnectionError, APIError) as e:
            last_err = e
            sleep_s = min(60.0, base * (2 ** (attempt - 1)) + float(np.random.uniform(0, 1.0)))
            print(f"TRANSIENT API dialog={dialog_id} run={run_id} attempt={attempt}/{llm_cfg.max_retries} -> sleep {sleep_s:.1f}s | {type(e).__name__}")
            time.sleep(sleep_s)

        except Exception as e:
            last_err = e
            time.sleep(base)

    raise RuntimeError(f"LLM failed (dialog={dialog_id}, run={run_id}): {last_err!r}")

### Run full LLM evaluation (3 Runs for each dialogs) and median aggregation

In [ ]:
def run_llm_all_dialogs(resume: bool = True) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Evaluates all dialogs with 3 independent runs and computes medians per dialog×scale.
    Stores:
      - llm_runs_all_dialogs.csv   (dialog × run)
      - llm_median_all_dialogs.csv (dialog, median per scale)
    """
    stimuli = load_stimuli()

    runs_path = os.path.join(cfg.out_dir, "llm_runs_all_dialogs.csv")
    med_path  = os.path.join(cfg.out_dir, "llm_median_all_dialogs.csv")

    runs_existing = pd.read_csv(runs_path) if (resume and os.path.exists(runs_path)) else pd.DataFrame()
    if not runs_existing.empty:
        runs_existing[cfg.id_col] = runs_existing[cfg.id_col].astype(str).str.strip()
        runs_existing["run_id"] = pd.to_numeric(runs_existing["run_id"], errors="coerce")

    total_jobs = len(stimuli) * llm_cfg.n_runs
    done = 0
    t0 = time.time()

    new_rows = []

    for _, row in stimuli.iterrows():
        did = str(row[cfg.id_col]).strip()
        txt = str(row[cfg.text_col])

        for run_id in range(1, llm_cfg.n_runs + 1):
            # resume-safe skip
            if not runs_existing.empty:
                mask = (runs_existing[cfg.id_col] == did) & (runs_existing["run_id"] == run_id)
                if mask.any():
                    row_done = runs_existing.loc[mask].tail(1)
                    llm_cols = [f"llm_{k}" for k in SCALE_KEYS]
                    if all(c in row_done.columns for c in llm_cols):
                        vals = row_done[llm_cols].apply(pd.to_numeric, errors="coerce").to_numpy().ravel()
                        if np.isfinite(vals).all():
                            continue

            scores, raw = llm_rate_one(txt, did, run_id)

            out = {
                cfg.id_col: did,
                "run_id": run_id,
                "temperature": llm_cfg.temperatures[run_id - 1],
                "raw_response": raw,
                cfg.condition_col: row.get(cfg.condition_col, ""),
                cfg.is_gold_col: int(row.get(cfg.is_gold_col, 0)),
                cfg.is_imc_col: int(row.get(cfg.is_imc_col, 0)),
            }
            for k in SCALE_KEYS:
                out[f"llm_{k}"] = scores[k]
            new_rows.append(out)

            done += 1
            elapsed = (time.time() - t0) / 60.0
            eta = (elapsed / max(done, 1)) * (total_jobs - done)
            print(f"[LLM] {done}/{total_jobs} (dialog {did}, run {run_id}) | elapsed {elapsed:.1f}m | ETA {eta:.1f}m")

            runs_out = pd.concat([runs_existing, pd.DataFrame(new_rows)], ignore_index=True)
            runs_out.to_csv(runs_path, index=False, encoding="utf-8")
            runs_existing = runs_out
            new_rows = []

    runs_df = pd.read_csv(runs_path)
    runs_df[cfg.id_col] = runs_df[cfg.id_col].astype(str).str.strip()

    # median aggregation (requires exactly 3 runs per dialog)
    def _median3(g: pd.DataFrame) -> pd.Series:
        if len(g) != llm_cfg.n_runs:
            out = {f"llm_median_{k}": np.nan for k in SCALE_KEYS}
            out["n_runs"] = int(len(g))
            out["median_valid"] = 0
            return pd.Series(out)

        out = {}
        for k in SCALE_KEYS:
            vals = pd.to_numeric(g[f"llm_{k}"], errors="coerce")
            out[f"llm_median_{k}"] = float(np.median(vals))

        out["n_runs"] = int(len(g))
        out["median_valid"] = int(all(np.isfinite(out[f"llm_median_{k}"]) for k in SCALE_KEYS))
        return pd.Series(out)

    med = runs_df.groupby(cfg.id_col).apply(_median3).reset_index()
    bad = med[med["median_valid"] != 1]
    if not bad.empty:
        examples = bad[cfg.id_col].head(10).tolist()
        raise RuntimeError(f"Median aggregation invalid for some dialogs. Examples: {examples}")

    # attach stimulus meta (except full dialog text)
    stimuli = load_stimuli()
    meta_cols = [c for c in stimuli.columns if c != cfg.text_col]
    med_df = stimuli[meta_cols].merge(med, on=cfg.id_col, how="left")

    med_df.to_csv(med_path, index=False, encoding="utf-8")
    print("Saved:", runs_path)
    print("Saved:", med_path)
    return runs_df, med_df

### Export PRIMARY / SENS subsets for LLM (matching human gold sets)

In [ ]:
def export_llm_subsets_for_primary_and_sens() -> Dict[str, str]:
    """
    Exports LLM median subsets that matches the dialog sets available in:
    - human_PRIMARY_gold_dialog_level.csv
    - human_SENS_gold_dialog_level.csv
    """
    llm_med_path = os.path.join(cfg.out_dir, "llm_median_all_dialogs.csv")
    if not os.path.exists(llm_med_path):
        raise FileNotFoundError("Run run_llm_all_dialogs() first to create llm_median_all_dialogs.csv")

    llm_med = pd.read_csv(llm_med_path)
    llm_med[cfg.id_col] = llm_med[cfg.id_col].astype(str).str.strip()

    h_primary_path = os.path.join(cfg.out_dir, "human_PRIMARY_gold_dialog_level.csv")
    h_sens_path    = os.path.join(cfg.out_dir, "human_SENS_gold_dialog_level.csv")
    if not os.path.exists(h_primary_path) or not os.path.exists(h_sens_path):
        raise FileNotFoundError("Run the Human export first (build_and_export_human_datasets).")

    h_primary = pd.read_csv(h_primary_path)
    h_sens    = pd.read_csv(h_sens_path)
    set_primary = set(h_primary[cfg.id_col].astype(str))
    set_sens    = set(h_sens[cfg.id_col].astype(str))

    llm_primary = llm_med[llm_med[cfg.id_col].isin(set_primary)].copy()
    llm_sens    = llm_med[llm_med[cfg.id_col].isin(set_sens)].copy()

    out1 = os.path.join(cfg.out_dir, "llm_PRIMARY_median_subset.csv")
    out2 = os.path.join(cfg.out_dir, "llm_SENS_median_subset.csv")
    llm_primary.to_csv(out1, index=False, encoding="utf-8")
    llm_sens.to_csv(out2, index=False, encoding="utf-8")

    print("\n LLM subset export summary ")
    print("LLM medians total dialogs:", llm_med[cfg.id_col].nunique())
    print("LLM PRIMARY subset dialogs:", llm_primary[cfg.id_col].nunique())
    print("LLM SENS subset dialogs:", llm_sens[cfg.id_col].nunique())

    return {"LLM_PRIMARY_subset": out1, "LLM_SENS_subset": out2}

### RUN (LLM)

In [ ]:
# Run LLM Evaluation:
runs_df, med_df = run_llm_all_dialogs(resume=True)

[LLM] 1/240 (dialog D067, run 3) | elapsed 0.1m | ETA 15.6m
[LLM] 2/240 (dialog D068, run 1) | elapsed 0.1m | ETA 13.5m
[LLM] 3/240 (dialog D068, run 2) | elapsed 0.2m | ETA 12.7m
[LLM] 4/240 (dialog D068, run 3) | elapsed 0.2m | ETA 12.4m
[LLM] 5/240 (dialog D069, run 1) | elapsed 0.3m | ETA 12.2m
[LLM] 6/240 (dialog D069, run 2) | elapsed 0.3m | ETA 12.1m
[LLM] 7/240 (dialog D069, run 3) | elapsed 0.4m | ETA 12.2m
[LLM] 8/240 (dialog D070, run 1) | elapsed 0.4m | ETA 12.1m
[LLM] 9/240 (dialog D070, run 2) | elapsed 0.5m | ETA 11.9m
[LLM] 10/240 (dialog D070, run 3) | elapsed 0.5m | ETA 11.7m
[LLM] 11/240 (dialog D071, run 1) | elapsed 0.6m | ETA 11.6m
[LLM] 12/240 (dialog D071, run 2) | elapsed 0.6m | ETA 11.4m
[LLM] 13/240 (dialog D071, run 3) | elapsed 0.7m | ETA 11.4m
[LLM] 14/240 (dialog D072, run 1) | elapsed 0.7m | ETA 11.3m
[LLM] 15/240 (dialog D072, run 2) | elapsed 0.7m | ETA 11.2m
[LLM] 16/240 (dialog D072, run 3) | elapsed 0.8m | ETA 11.1m
[LLM] 17/240 (dialog D073, run 1)

/tmp/ipython-input-3387676526.py:85: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  med = runs_df.groupby(cfg.id_col).apply(_median3).reset_index()


In [ ]:
# export subsets that match human gold dialog sets:
llm_exports = export_llm_subsets_for_primary_and_sens()
llm_exports


 LLM subset export summary 
LLM medians total dialogs: 80
LLM PRIMARY subset dialogs: 50
LLM SENS subset dialogs: 54


{'LLM_PRIMARY_subset': '/content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/studyA_exports/llm_PRIMARY_median_subset.csv',
 'LLM_SENS_subset': '/content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/studyA_exports/llm_SENS_median_subset.csv'}

## Study A - Hypotheses Overview (no tests executed here)

In [ ]:
import numpy as np
import pandas as pd
import math

### Hypotheses Overview table

In [ ]:
def build_hypotheses_overview_table() -> pd.DataFrame:
    """
    Provides a prereg-aligned hypothesis overview (what data, what metric, what benchmark).
    Does not run tests, this is only the analysis map.
    """
    rows = [
        # Confirmatory
        {"H": "H1", "Goal": "Criterion validity (overall)", "Dataset": "PRIMARY (+ SENS repeat)",
         "Inputs": "Human gold overall vs LLM median overall", "Planned metric": "Pearson r + Spearman rho", "Benchmark": "r ≥ .40",
         "Inferential?": "Yes (later)", "Implementation": "Python (scipy/statsmodels)"},
        {"H": "H2", "Goal": "Criterion validity (7 subscales)", "Dataset": "PRIMARY (+ SENS repeat)",
         "Inputs": "Human gold subscales vs LLM median subscales", "Planned metric": "Pearson r / Spearman rho + BH-FDR across 7", "Benchmark": "positive; ideally r ≥ .40",
         "Inferential?": "Yes (later)", "Implementation": "Python (scipy + statsmodels FDR)"},
        {"H": "H3", "Goal": "Human interrater reliability", "Dataset": "PRIMARY humans (regular rows; 2 raters/dialog)",
         "Inputs": "Two human raters per dialog", "Planned metric": "ICC(2,2)", "Benchmark": "ICC ≥ .75",
         "Inferential?": "Yes (later)", "Implementation": "R or Python (pingouin); R often preferred for reporting"},
        {"H": "H4", "Goal": "Human discriminability (good vs bad)", "Dataset": "PRIMARY human gold",
         "Inputs": "Human gold overall + condition", "Planned metric": "ROC-AUC + Cohen’s d", "Benchmark": "AUC ≥ .70; d ≥ .30",
         "Inferential?": "Yes (later)", "Implementation": "Python (sklearn + custom d)"},
        {"H": "H5", "Goal": "LLM discriminability (good vs bad)", "Dataset": "PRIMARY LLM medians",
         "Inputs": "LLM median overall + condition", "Planned metric": "ROC-AUC + Cohen’s d", "Benchmark": "AUC ≥ .70; d ≥ .30",
         "Inferential?": "Yes (later)", "Implementation": "Python (sklearn + custom d)"},

        # Secondary / robustness (prereg)
        {"H": "H6", "Goal": "Convergent validity (rule-metrics vs human)", "Dataset": "SECONDARY/Exploratory",
         "Inputs": "Rule-based submetrics vs human gold", "Planned metric": "Correlations + correction within families", "Benchmark": "descriptive",
         "Inferential?": "Exploratory", "Implementation": "Python or R"},
        {"H": "H7", "Goal": "Convergent validity (rule-metrics vs LLM)", "Dataset": "SECONDARY/Exploratory",
         "Inputs": "Rule-based submetrics vs LLM medians", "Planned metric": "Correlations + correction within families", "Benchmark": "descriptive",
         "Inferential?": "Exploratory", "Implementation": "Python or R"},
        {"H": "H8", "Goal": "Bias/agreement beyond correlation", "Dataset": "PRIMARY (+ SENS repeat)",
         "Inputs": "LLM median overall vs human gold overall", "Planned metric": "Bland–Altman bias + LoA (bootstrap CI)", "Benchmark": "|bias| ≤ .50 Likert points",
         "Inferential?": "Yes (later)", "Implementation": "Python (numpy + bootstrap)"},
        {"H": "H9", "Goal": "Robustness to text features (post-hoc)", "Dataset": "SECONDARY/Robustness",
         "Inputs": "Controls: length/readability etc.", "Planned metric": "partial correlations / regression checks", "Benchmark": "descriptive robustness",
         "Inferential?": "Robustness", "Implementation": "R often convenient; Python also possible"},
    ]
    return pd.DataFrame(rows)

### Feasibility helpers (no tests, just CI-width planning)

In [ ]:
def corr_ci_width_95(r: float, n: int) -> float:
    """
    Approximates the 95% CI width for Pearson r via Fisher-z (planning tool).
    This is not a hypothesis test, it just quantifies precision at given n.
    """
    if n <= 3:
        return np.nan
    r = max(min(r, 0.999999), -0.999999)
    z = np.arctanh(r)
    se = 1.0 / math.sqrt(n - 3)
    z_lo = z - 1.96 * se
    z_hi = z + 1.96 * se
    r_lo = np.tanh(z_lo)
    r_hi = np.tanh(z_hi)
    return float(r_hi - r_lo)


def feasibility_report(n_dialogs: int, target_r: float = 0.40) -> pd.DataFrame:
    """
    Shows how precise the correlation estimate is at n dialogs (CI width),
    using the prereg benchmark r>=.40 as reference.
    """
    width = corr_ci_width_95(target_r, n_dialogs)
    return pd.DataFrame([{
        "n_dialogs": n_dialogs,
        "benchmark_r": target_r,
        "approx_95CI_width_for_r_at_benchmark": width,
        "interpretation": "Smaller width = more precise estimate, this is precision planning, not power."
    }])

### RUN (Hypotheses overview)

In [ ]:
hyp_tbl = build_hypotheses_overview_table()
display(hyp_tbl)

# Example: feasibility for n=46 vs n=50 vs n=80 (you can change these)
display(pd.concat([
    feasibility_report(46, 0.40),
    feasibility_report(50, 0.40),
    feasibility_report(80, 0.40),
], ignore_index=True))

,H,Goal,Dataset,Inputs,Planned metric,Benchmark,Inferential?,Implementation
0,H1,Criterion validity (overall),PRIMARY (+ SENS repeat),Human gold overall vs LLM median overall,Pearson r + Spearman rho,r ≥ .40,Yes (later),Python (scipy/statsmodels)
1,H2,Criterion validity (7 subscales),PRIMARY (+ SENS repeat),Human gold subscales vs LLM median subscales,Pearson r / Spearman rho + BH-FDR across 7,positive; ideally r ≥ .40,Yes (later),Python (scipy + statsmodels FDR)
2,H3,Human interrater reliability,PRIMARY humans (regular rows; 2 raters/dialog),Two human raters per dialog,"ICC(2,2)",ICC ≥ .75,Yes (later),R or Python (pingouin); R often preferred for ...
3,H4,Human discriminability (good vs bad),PRIMARY human gold,Human gold overall + condition,ROC-AUC + Cohen’s d,AUC ≥ .70; d ≥ .30,Yes (later),Python (sklearn + custom d)
4,H5,LLM discriminability (good vs bad),PRIMARY LLM medians,LLM median overall + condition,ROC-AUC + Cohen’s d,AUC ≥ .70; d ≥ .30,Yes (later),Python (sklearn + custom d)
5,H6,Convergent validity (rule-metrics vs human),SECONDARY/Exploratory,Rule-based submetrics vs human gold,Correlations + correction within families,descriptive,Exploratory,Python or R
6,H7,Convergent validity (rule-metrics vs LLM),SECONDARY/Exploratory,Rule-based submetrics vs LLM medians,Correlations + correction within families,descriptive,Exploratory,Python or R
7,H8,Bias/agreement beyond correlation,PRIMARY (+ SENS repeat),LLM median overall vs human gold overall,Bland–Altman bias + LoA (bootstrap CI),|bias| ≤ .50 Likert points,Yes (later),Python (numpy + bootstrap)
8,H9,Robustness to text features (post-hoc),SECONDARY/Robustness,Controls: length/readability etc.,partial correlations / regression checks,descriptive robustness,Robustness,R often convenient; Python also possible


,n_dialogs,benchmark_r,approx_95CI_width_for_r_at_benchmark,interpretation
0,46,0.4,0.494375,"Smaller width = more precise estimate, this is..."
1,50,0.4,0.473502,"Smaller width = more precise estimate, this is..."
2,80,0.4,0.372004,"Smaller width = more precise estimate, this is..."


## Dataset Inventory (PRIMARY vs. SENS)

In [ ]:
# Dataset Inventory (PRIMARY vs SENS), no stats, just coverage
# Requires: the HUMAN export step has been run (build_and_export_human_datasets)
# Optional: report LLM subsets too

import os
import pandas as pd

def dataset_inventory(cfg) -> Dict[str, pd.DataFrame]:
    """
    Loads the exported CSVs and prints a compact inventory:
    - counts (dialogs / rows / raters)
    - condition split (good/bad) where available
    - missingness: dialogs in stimuli but not in PRIMARY/SENS human-gold
    - top exclusion reasons (row-level)
    - excluded raters count + top reasons (rater-level)
    """
    out = {}

    # Load stimuli meta (ground truth frame of reference)
    stim = load_stimuli()
    # pull the exact column names expected; fall back if cfg.* mismatches
    idcol = cfg.id_col
    condcol = cfg.condition_col if cfg.condition_col in stim.columns else "condition"
    goldcol = cfg.is_gold_col if cfg.is_gold_col in stim.columns else "is_gold"
    imccol  = cfg.is_imc_col if cfg.is_imc_col in stim.columns else "is_imc"

    missing = [c for c in [idcol, condcol, goldcol, imccol] if c not in stim.columns]
    if missing:
      raise KeyError(f"Stimuli missing required columns for inventory: {missing}. Found: {list(stim.columns)}")

    stim_meta = stim[[idcol, condcol, goldcol, imccol]].copy()
    stim_meta[idcol] = stim_meta[idcol].astype(str).str.strip()
    stim_meta[condcol] = stim_meta[condcol].astype(str).str.strip().str.lower()

    def _read(path):
        return pd.read_csv(path) if os.path.exists(path) else None

    # Expected exports from HUMAN block
    p_rows = _read(os.path.join(cfg.out_dir, "human_PRIMARY_rows_regular_only_kept.csv"))
    p_gold = _read(os.path.join(cfg.out_dir, "human_PRIMARY_gold_dialog_level.csv"))
    p_aud_rows = _read(os.path.join(cfg.out_dir, "audit_PRIMARY_row_level.csv"))
    p_aud_raters = _read(os.path.join(cfg.out_dir, "audit_PRIMARY_rater_level.csv"))
    p_aud_dialogs = _read(os.path.join(cfg.out_dir, "audit_PRIMARY_dialog_level.csv"))

    s_rows = _read(os.path.join(cfg.out_dir, "human_SENS_rows_regular_only_kept.csv"))
    s_gold = _read(os.path.join(cfg.out_dir, "human_SENS_gold_dialog_level.csv"))
    s_aud_rows = _read(os.path.join(cfg.out_dir, "audit_SENS_row_level.csv"))
    s_aud_raters = _read(os.path.join(cfg.out_dir, "audit_SENS_rater_level.csv"))
    s_aud_dialogs = _read(os.path.join(cfg.out_dir, "audit_SENS_dialog_level.csv"))

    # Optional exports from LLM block
    llm_all = _read(os.path.join(cfg.out_dir, "llm_median_all_dialogs.csv"))
    llm_p = _read(os.path.join(cfg.out_dir, "llm_PRIMARY_median_subset.csv"))
    llm_s = _read(os.path.join(cfg.out_dir, "llm_SENS_median_subset.csv"))

    # Helper: safe condition counts
    def _cond_counts(df):
      if df is None:
        return None

      if idcol not in df.columns:
        raise KeyError(f"Expected id column '{idcol}' not found in df columns: {list(df.columns)}")

      tmp = df.copy()
      tmp[idcol] = tmp[idcol].astype(str).str.strip()

      # If condition present in df, use it (else merge from stimuli)
      if condcol in tmp.columns:
        cc = tmp[condcol].astype(str).str.strip().str.lower()
      else:
        m = tmp.merge(stim_meta[[idcol, condcol]], on=idcol, how="left")
        cc = m[condcol].astype(str).str.strip().str.lower()

      vc = cc.value_counts(dropna=False).rename_axis("condition").reset_index(name="n_dialogs")
      return vc

    def _missing_dialogs(df_gold):
        if df_gold is None:
            return None
        all_regular = stim_meta[(stim_meta[cfg.is_gold_col] == 0) & (stim_meta[cfg.is_imc_col] == 0)][cfg.id_col].astype(str)
        have = set(df_gold[cfg.id_col].astype(str))
        miss = sorted(set(all_regular) - have)
        return pd.DataFrame({cfg.id_col: miss}).merge(stim_meta[[cfg.id_col, cfg.condition_col]], on=cfg.id_col, how="left")

    def _top_reasons(aud_rows, topk=15):
        if aud_rows is None or "exclusion_reasons" not in aud_rows.columns or "keep_row" not in aud_rows.columns:
            return None
        ex = aud_rows[aud_rows["keep_row"] == False].copy()
        if ex.empty:
            return pd.DataFrame(columns=["reason", "n_rows"])
        # explode comma-separated reasons
        ex["reason_list"] = ex["exclusion_reasons"].fillna("").astype(str).apply(
            lambda s: [r.strip() for r in s.split(",") if r.strip()]
        )
        exploded = ex.explode("reason_list")
        vc = exploded["reason_list"].value_counts().head(topk).rename_axis("reason").reset_index(name="n_rows")
        return vc

    def _excluded_raters(aud_raters, topk=15):
        if aud_raters is None or "status" not in aud_raters.columns:
            return None
        ex = aud_raters[aud_raters["status"] == "EXCLUDED"].copy()
        summary = pd.DataFrame([{
            "n_excluded_raters": int(len(ex)),
            "n_kept_raters": int((aud_raters["status"] == "KEPT").sum()),
            "n_total_raters": int(len(aud_raters))
        }])
        # top compact reasons
        if "reason_compact" in ex.columns:
            vc = ex["reason_compact"].fillna("(none)").value_counts().head(topk).rename_axis("reason_compact").reset_index(name="n_raters")
        else:
            vc = None
        return summary, vc, ex

    # PRIMARY inventory

    print("DATASET INVENTORY — PRIMARY (Human)")

    if p_rows is not None:
        print(f"PRIMARY rows kept (regular only): {len(p_rows)}")
        print(f"PRIMARY raters (in kept rows):    {p_rows[cfg.rater_col].nunique() if cfg.rater_col in p_rows.columns else 'n/a'}")
        print(f"PRIMARY dialogs (in kept rows):   {p_rows[cfg.id_col].nunique()}")

    if p_gold is not None:
        print(f"PRIMARY human gold dialogs:       {p_gold[cfg.id_col].nunique()}")
        out["primary_condition_dialogs"] = _cond_counts(p_gold)
        display(out["primary_condition_dialogs"])

        miss_p = _missing_dialogs(p_gold)
        out["primary_missing_dialogs"] = miss_p
        print(f"Missing regular dialogs vs stimuli (PRIMARY gold): {len(miss_p)}")
        display(miss_p.head(25))

    if p_aud_rows is not None:
        top_p = _top_reasons(p_aud_rows)
        out["primary_top_row_exclusion_reasons"] = top_p
        print("\nTop row-level exclusion reasons (PRIMARY):")
        display(top_p)

    if p_aud_raters is not None:
        summ, top_rr, ex_raters = _excluded_raters(p_aud_raters)
        out["primary_rater_exclusion_summary"] = summ
        out["primary_top_rater_reasons"] = top_rr
        out["primary_excluded_raters_table"] = ex_raters
        print("\nRater exclusion summary (PRIMARY):")
        display(summ)
        print("\nTop excluded-rater reasons (PRIMARY):")
        display(top_rr)
        print("\nExcluded raters (PRIMARY) — table:")
        display(ex_raters.head(50))

    if p_aud_dialogs is not None and "kept_for_human_gold" in p_aud_dialogs.columns:
        # show most common dialog exclusion reasons
        ex = p_aud_dialogs[p_aud_dialogs["kept_for_human_gold"] == False].copy()
        print("\nDialogs excluded from PRIMARY human-gold (count):", len(ex))
        if "reason_compact" in ex.columns:
            vc = ex["reason_compact"].value_counts().head(20).rename_axis("reason_compact").reset_index(name="n_dialogs")
            out["primary_top_dialog_exclusion_reasons"] = vc
            print("Top dialog-level exclusion reasons (PRIMARY):")
            display(vc)

    # SENS inventory

    print("DATASET INVENTORY — SENSITIVITY (Human)")

    if s_rows is not None:
        print(f"SENS rows kept (regular only): {len(s_rows)}")
        print(f"SENS raters (in kept rows):    {s_rows[cfg.rater_col].nunique() if cfg.rater_col in s_rows.columns else 'n/a'}")
        print(f"SENS dialogs (in kept rows):   {s_rows[cfg.id_col].nunique()}")

    if s_gold is not None:
        print(f"SENS human gold dialogs:       {s_gold[cfg.id_col].nunique()}")
        out["sens_condition_dialogs"] = _cond_counts(s_gold)
        display(out["sens_condition_dialogs"])

        miss_s = _missing_dialogs(s_gold)
        out["sens_missing_dialogs"] = miss_s
        print(f"Missing regular dialogs vs stimuli (SENS gold): {len(miss_s)}")
        display(miss_s.head(25))

    if s_aud_rows is not None:
        top_s = _top_reasons(s_aud_rows)
        out["sens_top_row_exclusion_reasons"] = top_s
        print("\nTop row-level exclusion reasons (SENS):")
        display(top_s)

    if s_aud_raters is not None:
        summ, top_rr, ex_raters = _excluded_raters(s_aud_raters)
        out["sens_rater_exclusion_summary"] = summ
        out["sens_top_rater_reasons"] = top_rr
        out["sens_excluded_raters_table"] = ex_raters
        print("\nRater exclusion summary (SENS):")
        display(summ)
        print("\nTop excluded-rater reasons (SENS):")
        display(top_rr)

    if s_aud_dialogs is not None and "kept_for_human_gold" in s_aud_dialogs.columns:
        ex = s_aud_dialogs[s_aud_dialogs["kept_for_human_gold"] == False].copy()
        print("\nDialogs excluded from SENS human-gold (count):", len(ex))
        if "reason_compact" in ex.columns:
            vc = ex["reason_compact"].value_counts().head(20).rename_axis("reason_compact").reset_index(name="n_dialogs")
            out["sens_top_dialog_exclusion_reasons"] = vc
            print("Top dialog-level exclusion reasons (SENS):")
            display(vc)

    # LLM inventory (optional)

    print("Dataset Inventory — LLM (optional, if exported)")

    if llm_all is not None:
        print("LLM medians (all dialogs):", llm_all[cfg.id_col].nunique())
        display(_cond_counts(llm_all))

    if llm_p is not None:
        print("LLM PRIMARY subset dialogs:", llm_p[cfg.id_col].nunique())
        display(_cond_counts(llm_all))
    else:
        print("LLM PRIMARY subset not found (run export_llm_subsets_for_primary_and_sens).")

    if llm_s is not None:
        print("LLM SENS subset dialogs:", llm_s[cfg.id_col].nunique())
        display(_cond_counts(llm_s))
    else:
        print("LLM SENS subset not found (run export_llm_subsets_for_primary_and_sens).")

    return out

### RUN (inventory)

In [ ]:
inventory = dataset_inventory(cfg)
inventory.keys()

DATASET INVENTORY — PRIMARY (Human)
PRIMARY rows kept (regular only): 104
PRIMARY raters (in kept rows):    13
PRIMARY dialogs (in kept rows):   54
PRIMARY human gold dialogs:       50


,condition,n_dialogs
0,bad,26
1,good,24


Missing regular dialogs vs stimuli (PRIMARY gold): 4


,dialog_id,condition
0,D008,good
1,D013,good
2,D020,good
3,D049,bad



Top row-level exclusion reasons (PRIMARY):


...

LLM PRIMARY subset dialogs: 50


,condition,n_dialogs
0,good,40
1,bad,40


LLM SENS subset dialogs: 54


,condition,n_dialogs
0,good,27
1,bad,27


dict_keys(['primary_condition_dialogs', 'primary_missing_dialogs', 'primary_top_row_exclusion_reasons', 'primary_rater_exclusion_summary', 'primary_top_rater_reasons', 'primary_excluded_raters_table', 'primary_top_dialog_exclusion_reasons', 'sens_condition_dialogs', 'sens_missing_dialogs', 'sens_top_row_exclusion_reasons', 'sens_rater_exclusion_summary', 'sens_top_rater_reasons', 'sens_excluded_raters_table', 'sens_top_dialog_exclusion_reasons'])